<a href="https://colab.research.google.com/github/vsindrajith7/predictive-project-grp-3/blob/main/preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import re
import pickle
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [18]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [19]:
CSV_PATH     = "emails.csv"      # path to your dataset
TEXT_COLUMN  = "text"            # column that holds email body
LABEL_COLUMN = "spam"           # column that holds spam/ham label  (spam=1, ham=0)
MAX_WORDS    = 10000             # vocabulary size for LSTM tokenizer
MAX_LEN      = 200               # max sequence length for padding
TEST_SIZE    = 0.2
RANDOM_STATE = 42

Load the dataset

In [20]:
print("Loading dataset...")
df = pd.read_csv(CSV_PATH)
print(f"  Shape : {df.shape}")
print(f"  Columns: {df.columns.tolist()}")

# Convert label to binary if needed  (spam→1, ham→0)
if df[LABEL_COLUMN].dtype == object:
    df[LABEL_COLUMN] = df[LABEL_COLUMN].map({'spam': 1, 'ham': 0})

Loading dataset...
  Shape : (5728, 2)
  Columns: ['text', 'spam']


Preprocessing

In [21]:
lemmatizer  = WordNetLemmatizer()
stop_words  = set(stopwords.words('english'))

def clean_text(text):
  # Ensure text is a string
    text = str(text)
    # Remove HTML tags, URLs, email addresses, and special characters/digits
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Convert to lowercase
    text = text.lower()
    # Tokenize
    tokens = nltk.word_tokenize(text)
    # Remove stopwords and lemmatize
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words and word.isalpha()]
    return " ".join(cleaned_tokens)

print("\nCleaning email text...")
df['cleaned'] = df[TEXT_COLUMN].apply(clean_text)
print("  Done.")


Cleaning email text...
  Done.


Train / Test Split

In [22]:
X = df['cleaned'].values
y = df[LABEL_COLUMN].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f"\nTrain size : {len(X_train)}")
print(f"Test  size : {len(X_test)}")


Train size : 4582
Test  size : 1146


TF-IDF Features  (for SVM)

In [24]:
print("\nBuilding TF-IDF features...")
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)
print(f"  TF-IDF shape (train): {X_train_tfidf.shape}")

with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print("  Saved: tfidf_vectorizer.pkl")


Building TF-IDF features...
  TF-IDF shape (train): (4582, 10000)
  Saved: tfidf_vectorizer.pkl


Padded Sequences  (for LSTM)

In [28]:
print("\nBuilding word sequences for LSTM...")
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN, padding='post')
X_test_seq  = pad_sequences(tokenizer.texts_to_sequences(X_test),  maxlen=MAX_LEN, padding='post')
print(f"  Sequence shape (train): {X_train_seq.shape}")

with open('lstm_tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
print("  Saved: lstm_tokenizer.pkl")


Building word sequences for LSTM...
  Sequence shape (train): (4582, 200)
  Saved: lstm_tokenizer.pkl
